# T3 (palletizing) data collection

Healthy and anomaly recording via RTDE at 125 Hz. Payload 3 kg.

Includes cycle-segmentation fix for the 4-pallet trajectory and the duct-tape rerun sequence.

**Paper:** Zafar SA, Qin W. *Cross-task anomaly detection in reconfigurable industrial robot systems based on physics-structured regression of joint motor currents* (2026).


In [ ]:
# Payload: ~3 kg physical, 3.0 kg pendant

from rtde_receive import RTDEReceiveInterface

ROBOT_IP = "192.168.35.82"
rtde_r = RTDEReceiveInterface(ROBOT_IP)
print("Connected:", rtde_r.isConnected())

tcp0 = rtde_r.getActualTCPPose()
print(f"TCP: x={tcp0[0]:.3f}  y={tcp0[1]:.3f}  z={tcp0[2]:.3f}")
print(f"T3 reaches 18cm forward, 15cm back, 15cm sideways, 8cm down from HOME.")
print(f"Verify 20cm clearance in all directions.")

import socket, time, json, csv
import numpy as np
import h5py
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime

matplotlib.rcParams.update({
    "font.family":       "Arial",
    "font.size":         6,
    "axes.titlesize":    7,
    "axes.labelsize":    6,
    "xtick.labelsize":   5,
    "ytick.labelsize":   5,
    "legend.fontsize":   5,
    "figure.titlesize":  7,
    "axes.linewidth":    0.5,
    "xtick.major.width": 0.4,
    "ytick.major.width": 0.4,
    "xtick.major.size":  2,
    "ytick.major.size":  2,
    "lines.linewidth":   0.4,
    "savefig.dpi":       1200,
    "savefig.bbox":      "tight",
    "savefig.pad_inches": 0.02,
    "pdf.fonttype":      42,
    "ps.fonttype":       42,
})

FIG_SINGLE = (90/25.4, 70/25.4)
FIG_DOUBLE = (180/25.4, 70/25.4)
FIG_TALL   = (180/25.4, 140/25.4)

C_SIGNAL = "#D55E00"
C_TCP    = "#0072B2"
C_HOME   = "#009E73"
C_FAR    = "#CC79A7"

ROOT        = Path(r"D:\Research\RCM")
LAB_DATA    = ROOT / "Lab_Data" / "T3_Palletize" / "healthy"
LOGS_DIR    = ROOT / "Logs"
DATASET_LOG = ROOT / "Lab_Data" / "dataset_log.csv"
FIG_SUP     = ROOT / "Manuscript_Data" / "Figures" / "Supplementary"
for d in [LAB_DATA, LOGS_DIR, FIG_SUP]:
    d.mkdir(parents=True, exist_ok=True)

NOTEBOOK = "NB3_T3_Healthy"
TASK     = "T3"
HZ       = 125
N_SCRIPT = 180
N_ACCEPT = 160

PAYLOAD_PENDANT_KG  = 3.0
PAYLOAD_PHYSICAL_KG = 3.0

# T3: palletizing — moderate speed, extended reach, 4 pallet positions
V     = 0.06
A     = 0.25
SLEEP = 0.3

SCRIPT_PORT = 30002

def send_urscript(script: str):
    if not script.endswith("\n"):
        script += "\n"
    with socket.create_connection((ROBOT_IP, SCRIPT_PORT), timeout=5) as s:
        s.sendall(script.encode("utf-8"))

def build_t3(N):
    return f"""
def task_t3():
  local v  = {V}
  local a  = {A}
  local dt = {SLEEP}
  local N  = {N}
  local i  = 0

  local home = get_actual_tcp_pose()

  local pick   = pose_trans(home, p[0.0, 0.0, -0.08, 0,0,0])
  local abpick = pose_trans(home, p[0.0, 0.0,  0.0,  0,0,0])

  local pal1 = pose_trans(home, p[ 0.18, -0.12, -0.05, 0,0,0])
  local pal2 = pose_trans(home, p[ 0.18,  0.12, -0.05, 0,0,0])
  local pal3 = pose_trans(home, p[-0.15, -0.15, -0.05, 0,0,0])
  local pal4 = pose_trans(home, p[-0.15,  0.15, -0.05, 0,0,0])

  local above_h = 0.06

  while (i < N):
    movel(pick, a=a, v=v)
    sleep(dt)
    movel(abpick, a=a, v=v)

    local ab1 = pose_trans(pal1, p[0,0,above_h,0,0,0])
    movel(ab1,  a=a, v=v)
    movel(pal1, a=a, v=v*0.7)
    sleep(dt)
    movel(ab1,  a=a, v=v)

    movel(abpick, a=a, v=v)
    movel(pick, a=a, v=v)
    sleep(dt)
    movel(abpick, a=a, v=v)

    local ab2 = pose_trans(pal2, p[0,0,above_h,0,0,0])
    movel(ab2,  a=a, v=v)
    movel(pal2, a=a, v=v*0.7)
    sleep(dt)
    movel(ab2,  a=a, v=v)

    movel(abpick, a=a, v=v)
    movel(pick, a=a, v=v)
    sleep(dt)
    movel(abpick, a=a, v=v)

    local ab3 = pose_trans(pal3, p[0,0,above_h,0,0,0])
    movel(ab3,  a=a, v=v)
    movel(pal3, a=a, v=v*0.7)
    sleep(dt)
    movel(ab3,  a=a, v=v)

    movel(abpick, a=a, v=v)
    movel(pick, a=a, v=v)
    sleep(dt)
    movel(abpick, a=a, v=v)

    local ab4 = pose_trans(pal4, p[0,0,above_h,0,0,0])
    movel(ab4,  a=a, v=v)
    movel(pal4, a=a, v=v*0.7)
    sleep(dt)
    movel(ab4,  a=a, v=v)

    movel(home, a=a, v=v)
    i = i + 1
  end
end
task_t3()
"""

def save_fig(fig, name):
    fig.savefig(FIG_SUP / f"{name}.png")
    fig.savefig(FIG_SUP / f"{name}.pdf")

print(f"Task: {TASK} | N_SCRIPT: {N_SCRIPT} | Accept >= {N_ACCEPT}")
print(f"Payload: {PAYLOAD_PHYSICAL_KG}kg physical, {PAYLOAD_PENDANT_KG}kg pendant")
print(f"Save to: {LAB_DATA}")

input(f"Collect {N_SCRIPT} cycles. Press Enter to start (E-STOP ready!)...")

dt = 1.0 / HZ
max_sec = N_SCRIPT * 50 + 120  # T3 ~35-40s/cycle with 4 pallet positions
nmax = int(max_sec * HZ)

t    = np.zeros(nmax)
q    = np.zeros((nmax, 6))
qd   = np.zeros((nmax, 6))
cur  = np.zeros((nmax, 6))
tcp  = np.zeros((nmax, 6))
qdd  = np.zeros((nmax, 6))
mom  = np.zeros((nmax, 6))
tcpf = np.zeros((nmax, 6))

print("Logging started; sending URScript in 1.5s...")
t0 = time.perf_counter()
time.sleep(1.5)
send_urscript(build_t3(N_SCRIPT))
print("URScript sent. Robot should move now.\n")

idle_ctr = 0
idle_need = int(HZ * 4)
min_run = 30.0
last_print = t0

i = 0
while i < nmax:
    now = time.perf_counter()
    t[i]   = now - t0
    q[i]   = rtde_r.getActualQ()
    qd[i]  = rtde_r.getActualQd()
    cur[i] = rtde_r.getActualCurrent()
    tcp[i] = rtde_r.getActualTCPPose()
    try:    qdd[i]  = rtde_r.getTargetQdd()
    except: pass
    try:    mom[i]  = rtde_r.getTargetMoment()
    except: pass
    try:    tcpf[i] = rtde_r.getActualTCPForce()
    except: pass

    if now - last_print > 15.0:
        elapsed = now - t0
        v_now = np.max(np.abs(qd[max(0, i-HZ):i+1]))
        i_now = np.max(np.abs(cur[max(0, i-HZ):i+1]))
        print(f"  {elapsed:6.1f}s | samples: {i+1:>7,} | v_max: {v_now:.4f} | I_max: {i_now:.2f}A")
        last_print = now

    if t[i] > min_run:
        if np.max(np.abs(qd[i])) < 0.001:
            idle_ctr += 1
        else:
            idle_ctr = 0
        if idle_ctr >= idle_need:
            i += 1
            break

    target = (i + 1) * dt
    while (time.perf_counter() - t0) < target:
        pass
    i += 1

time.sleep(1)
send_urscript("stopj(2.0)\n")
print("Stop command sent.")

t, q, qd, cur, tcp = t[:i], q[:i], qd[:i], cur[:i], tcp[:i]
qdd, mom, tcpf = qdd[:i], mom[:i], tcpf[:i]

print(f"\nCollected {i:,} samples in {t[-1]:.1f}s ({i/t[-1]:.0f} Hz)")
print(f"v_max = {np.max(np.abs(qd)):.4f} rad/s | I_max = {np.max(np.abs(cur)):.3f} A")

home = tcp[0, :3]
dist = np.sqrt(np.sum((tcp[:, :3] - home)**2, axis=1)) * 1000

n_base = min(int(2.0 * HZ), len(dist) // 10)
noise = np.median(np.abs(np.diff(dist[:n_base]))) if n_base > 10 else 0.5
home_r = max(10.0, 3.0 * noise)
far_r  = 30.0

min_cycle_sec = 30.0  # real T3 cycle ~51s, sub-trips ~7s

cycle_num = np.zeros(len(tcp), dtype=np.int32)
c = 0
was_far = False
last_cycle_time = t[0]

for j in range(len(tcp)):
    if dist[j] > far_r:
        was_far = True
    if dist[j] < home_r and was_far:
        if (t[j] - last_cycle_time) >= min_cycle_sec:
            c += 1
            last_cycle_time = t[j]
        was_far = False
    cycle_num[j] = c

print(f"Detected {c} cycles (accept >= {N_ACCEPT}), home radius: {home_r:.1f}mm")

has_qdd  = bool(np.any(qdd != 0))
has_mom  = bool(np.any(mom != 0))
has_tcpf = bool(np.any(tcpf != 0))
print(f"target_qdd: {'available' if has_qdd else 'zeros (CB3 limitation)'}")
print(f"target_moment: {'available' if has_mom else 'zeros (CB3 limitation)'}")
print(f"actual_TCP_force: {'available' if has_tcpf else 'zeros (no F/T sensor)'}")

ts_str = datetime.now().strftime("%Y%m%d_%H%M%S")
fname = f"UR5_T3_healthy_{c}cyc_{HZ}hz_{ts_str}.h5"
fpath = LAB_DATA / fname

with h5py.File(str(fpath), "w") as f:
    for name, arr in [("timestamp", t), ("actual_q", q), ("actual_qd", qd),
                      ("actual_current", cur), ("actual_TCP_pose", tcp),
                      ("target_qdd", qdd), ("target_moment", mom),
                      ("actual_TCP_force", tcpf),
                      ("cycle_number", cycle_num)]:
        f.create_dataset(name, data=arr, compression="gzip")
    f.attrs.update({
        "task": TASK, "condition": "healthy",
        "n_script": N_SCRIPT, "n_detected": c,
        "hz": HZ, "duration_sec": float(t[-1]),
        "payload_pendant_kg": PAYLOAD_PENDANT_KG,
        "payload_physical_kg": PAYLOAD_PHYSICAL_KG,
        "v": V, "a": A, "notebook": NOTEBOOK,
        "has_target_qdd": has_qdd,
        "has_target_moment": has_mom,
        "has_actual_TCP_force": has_tcpf,
        "home_radius_mm": float(home_r),
        "min_cycle_duration_sec": min_cycle_sec,
    })

print(f"Saved: {fpath} ({fpath.stat().st_size/1024/1024:.1f} MB)")

log = {
    "notebook": NOTEBOOK, "timestamp": datetime.now().isoformat(),
    "task": TASK, "condition": "healthy",
    "n_script": N_SCRIPT, "n_detected": c, "hz": HZ,
    "duration_sec": round(float(t[-1]), 1),
    "payload_pendant_kg": PAYLOAD_PENDANT_KG,
    "payload_physical_kg": PAYLOAD_PHYSICAL_KG,
    "tcp_home": [round(x, 4) for x in tcp[0].tolist()],
    "robot_mode": int(rtde_r.getRobotMode()),
    "safety_mode": int(rtde_r.getSafetyMode()),
    "has_target_qdd": has_qdd, "has_target_moment": has_mom,
    "has_actual_TCP_force": has_tcpf,
    "file": fname,
}
log_path = LOGS_DIR / f"{NOTEBOOK}_{ts_str}.json"
with open(log_path, "w") as f:
    json.dump(log, f, indent=2)

csv_exists = DATASET_LOG.exists()
with open(DATASET_LOG, "a", newline="") as f:
    w = csv.writer(f)
    if not csv_exists:
        w.writerow(["filename","task","condition","anomaly","n_script","n_detected",
                     "duration_s","hz","payload_pendant_kg","payload_physical_kg",
                     "date","notebook","pass_fail"])
    w.writerow([fname, TASK, "healthy", "none", N_SCRIPT, c,
                round(t[-1],1), HZ, PAYLOAD_PENDANT_KG, PAYLOAD_PHYSICAL_KG,
                datetime.now().strftime("%Y-%m-%d %H:%M"), NOTEBOOK,
                "PASS" if c >= N_ACCEPT else "FAIL"])

JOINT = ["Base (J0)","Shoulder (J1)","Elbow (J2)","Wrist 1 (J3)","Wrist 2 (J4)","Wrist 3 (J5)"]
t_rel = t - t[0]

print(f"Duration:  {t_rel[-1]:.1f}s ({t_rel[-1]/60:.1f} min)")
print(f"Samples:   {len(t):,}")
print(f"Eff. Hz:   {len(t)/t_rel[-1]:.0f}")
print(f"Cycles:    {c}")

dt_ms = np.diff(t_rel) * 1000
print(f"dt jitter: median={np.median(dt_ms):.2f}ms  p95={np.percentile(dt_ms,95):.2f}ms  "
      f"p99={np.percentile(dt_ms,99):.2f}ms  max={np.max(dt_ms):.2f}ms")

has_nan = any(np.any(np.isnan(arr)) for arr in [q, qd, cur])
print(f"NaN: {'YES' if has_nan else 'None'}")

print(f"\nJoint currents:")
for j in range(6):
    cc = cur[:, j]
    print(f"  {JOINT[j]:16s}: mean={np.mean(cc):+.3f}A  "
          f"std={np.std(cc):.3f}A  [{np.min(cc):+.2f}, {np.max(cc):+.2f}]A")

print(f"\nJoint ranges:")
for j in range(6):
    rng = np.degrees(np.max(q[:,j]) - np.min(q[:,j]))
    print(f"  {JOINT[j]:16s}: {rng:.1f} deg")

issues = []
if c < N_ACCEPT: issues.append(f"Cycles: {c} < {N_ACCEPT}")
if has_nan:       issues.append("NaN values")
active = sum(1 for j in range(6) if np.degrees(np.max(q[:,j])-np.min(q[:,j])) > 5)
if active < 3:    issues.append(f"Only {active} joints active")

if issues:
    print(f"\nFAIL: {'; '.join(issues)} -> rerun")
else:
    print(f"\nPASS: {c} cycles, {len(t)/t_rel[-1]:.0f} Hz, {active} active joints")

mask = t_rel < 120
fig, axes = plt.subplots(6, 1, figsize=FIG_TALL, sharex=True)
fig.suptitle("T3 healthy: joint motor currents (first 120 s)", fontweight="bold")
for j in range(6):
    axes[j].plot(t_rel[mask], cur[mask,j], lw=0.3, color=C_SIGNAL, alpha=0.8)
    axes[j].set_ylabel(f"{JOINT[j]}\n(A)")
    axes[j].grid(True, alpha=0.2, lw=0.3)
    axes[j].text(-0.08, 1.0, chr(97+j), transform=axes[j].transAxes,
                 fontsize=8, fontweight="bold", va="top")
axes[-1].set_xlabel("Time (s)")
plt.tight_layout(h_pad=0.3)
save_fig(fig, "FigS_T3_healthy_currents")
plt.close(fig)

fig, ax = plt.subplots(figsize=FIG_DOUBLE)
ax.plot(t_rel, dist, lw=0.2, alpha=0.7, color=C_TCP)
ax.axhline(home_r, color=C_HOME, ls="--", lw=0.5, alpha=0.8,
           label=f"Home radius: {home_r:.1f} mm")
ax.axhline(far_r, color=C_FAR, ls="--", lw=0.5, alpha=0.7,
           label=f"Far threshold: {far_r:.0f} mm")
ax.set_ylabel("Distance from home (mm)")
ax.set_xlabel("Time (s)")
ax.set_title(f"T3 healthy: TCP distance from home ({c} cycles detected)", fontweight="bold")
ax.legend(frameon=False)
ax.grid(True, alpha=0.2, lw=0.3)
plt.tight_layout()
save_fig(fig, "FigS_T3_healthy_cycle_detect")
plt.close(fig)

if c >= 3:
    fig, axes = plt.subplots(3, 2, figsize=FIG_TALL)
    fig.suptitle(f"T3 healthy: cycle overlay of motor currents ({min(30,c)} cycles)",
                 fontweight="bold")
    for j in range(6):
        ax = axes[j//2, j%2]
        for ci in range(1, min(31, c+1)):
            cmask = cycle_num == ci
            if np.sum(cmask) < 10: continue
            ax.plot(cur[cmask, j], alpha=0.15, lw=0.3)
        ax.set_title(JOINT[j])
        ax.set_ylabel("Current (A)")
        ax.set_xlabel("Sample index")
        ax.grid(True, alpha=0.2, lw=0.3)
        ax.text(-0.12, 1.0, chr(97+j), transform=ax.transAxes,
                fontsize=8, fontweight="bold", va="top")
    plt.tight_layout(h_pad=0.5, w_pad=0.5)
    save_fig(fig, "FigS_T3_healthy_cycle_overlay")
    plt.close(fig)

fig = plt.figure(figsize=FIG_SINGLE)
ax3 = fig.add_subplot(111, projection="3d")
ss = max(1, len(tcp)//5000)
ax3.plot(tcp[::ss,0]*100, tcp[::ss,1]*100, tcp[::ss,2]*100,
         lw=0.2, alpha=0.5, color=C_TCP)
ax3.scatter(*tcp[0,:3]*100, c=C_HOME, s=30, label="Home", zorder=5, edgecolors="none")
ax3.set_xlabel("x (cm)")
ax3.set_ylabel("y (cm)")
ax3.set_zlabel("z (cm)")
ax3.set_title("T3 healthy: TCP trajectory", fontweight="bold")
ax3.legend(frameon=False)
ax3.tick_params(pad=1)
plt.tight_layout()
save_fig(fig, "FigS_T3_healthy_tcp_3d")
plt.close(fig)

print(f"Figures saved: {FIG_SUP}")
print(f"Data saved: {fpath}")

In [ ]:
# T3 Cycle Fix v2 — run once (~5 seconds)

import numpy as np
import h5py
from pathlib import Path

fpath = Path(r"D:\Research\RCM\Lab_Data\T3_Palletize\healthy\UR5_T3_healthy_256cyc_125hz_20260214_201715.h5")

with h5py.File(str(fpath), "r") as f:
    tcp = f["actual_TCP_pose"][:]
    t   = f["timestamp"][:]
    hz  = float(f.attrs["hz"])

home = tcp[0, :3]
dist = np.sqrt(np.sum((tcp[:, :3] - home)**2, axis=1)) * 1000

n_base = min(int(2.0 * hz), len(dist) // 10)
noise = np.median(np.abs(np.diff(dist[:n_base]))) if n_base > 10 else 0.5
home_r = max(10.0, 3.0 * noise)
far_r  = 30.0
min_cycle_sec = 45.0  # real cycle ~51s, must be just under

cycle_num = np.zeros(len(tcp), dtype=np.int32)
c = 0
was_far = False
last_cycle_time = t[0]

for j in range(len(tcp)):
    if dist[j] > far_r:
        was_far = True
    if dist[j] < home_r and was_far:
        if (t[j] - last_cycle_time) >= min_cycle_sec:
            c += 1
            last_cycle_time = t[j]
        was_far = False
    cycle_num[j] = c

print(f"Before: 256 cycles")
print(f"After:  {c} cycles (min_cycle_duration={min_cycle_sec}s)")

with h5py.File(str(fpath), "a") as f:
    if "cycle_number" in f:
        del f["cycle_number"]
    f.create_dataset("cycle_number", data=cycle_num, compression="gzip")
    f.attrs["n_detected"] = c
    f.attrs["min_cycle_duration_sec"] = min_cycle_sec

new_fname = fpath.name.replace("256cyc", f"{c}cyc")
new_fpath = fpath.parent / new_fname
fpath.rename(new_fpath)
print(f"Renamed: {new_fpath}")

In [ ]:
# Baseline: 3 kg gripper, 3.0 kg pendant (same as NB3 healthy)
# Anomalies: A2 (added mass), A3 (friction — DUCT TAPE only), A5 (TCP offset)
# A1 skipped (proven undetectable)

from rtde_receive import RTDEReceiveInterface

ROBOT_IP = "192.168.35.82"
rtde_r = RTDEReceiveInterface(ROBOT_IP)
print("Connected:", rtde_r.isConnected())

tcp0 = rtde_r.getActualTCPPose()
print(f"TCP: x={tcp0[0]:.3f}  y={tcp0[1]:.3f}  z={tcp0[2]:.3f}")
print(f"T3 palletizing: 4 pallet positions, large workspace")
print(f"REMINDER: Use DUCT TAPE for A3, NOT electrical tape!")

import socket, time, json, csv
import numpy as np
import h5py
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime

matplotlib.rcParams.update({
    "font.family":       "Arial",
    "font.size":         6,
    "axes.titlesize":    7,
    "axes.labelsize":    6,
    "xtick.labelsize":   5,
    "ytick.labelsize":   5,
    "legend.fontsize":   5,
    "figure.titlesize":  7,
    "axes.linewidth":    0.5,
    "xtick.major.width": 0.4,
    "ytick.major.width": 0.4,
    "xtick.major.size":  2,
    "ytick.major.size":  2,
    "lines.linewidth":   0.4,
    "savefig.dpi":       1200,
    "savefig.bbox":      "tight",
    "savefig.pad_inches": 0.02,
    "pdf.fonttype":      42,
    "ps.fonttype":       42,
})

FIG_TALL = (180/25.4, 140/25.4)

ROOT        = Path(r"D:\Research\RCM")
LOGS_DIR    = ROOT / "Logs"
DATASET_LOG = ROOT / "Lab_Data" / "dataset_log.csv"
FIG_SUP     = ROOT / "Manuscript_Data" / "Figures" / "Supplementary"
for d in [LOGS_DIR, FIG_SUP]:
    d.mkdir(parents=True, exist_ok=True)

NOTEBOOK = "NB6_T3_Anomaly"
TASK     = "T3"
HZ       = 125
SCRIPT_PORT = 30002

V     = 0.06
A     = 0.25
SLEEP = 0.3
MIN_CYCLE_SEC = 45  # T3 returns to HOME 7× per logical cycle

def send_urscript(script: str):
    if not script.endswith("\n"):
        script += "\n"
    with socket.create_connection((ROBOT_IP, SCRIPT_PORT), timeout=5) as s:
        s.sendall(script.encode("utf-8"))

def build_t3(N):
    return f"""
def task_t3():
  local v  = {V}
  local a  = {A}
  local dt = {SLEEP}
  local N  = {N}
  local i  = 0

  local home = get_actual_tcp_pose()

  local pick_above = pose_trans(home, p[ 0.00,  0.00, -0.03, 0,0,0])
  local pick       = pose_trans(home, p[ 0.00,  0.00, -0.08, 0,0,0])

  local pal1 = pose_trans(home, p[ 0.18,  0.15, -0.05, 0,0,0])
  local pal2 = pose_trans(home, p[ 0.18, -0.15, -0.05, 0,0,0])
  local pal3 = pose_trans(home, p[-0.15,  0.15, -0.05, 0,0,0])
  local pal4 = pose_trans(home, p[-0.15, -0.15, -0.05, 0,0,0])

  local pal1_above = pose_trans(home, p[ 0.18,  0.15,  0.00, 0,0,0])
  local pal2_above = pose_trans(home, p[ 0.18, -0.15,  0.00, 0,0,0])
  local pal3_above = pose_trans(home, p[-0.15,  0.15,  0.00, 0,0,0])
  local pal4_above = pose_trans(home, p[-0.15, -0.15,  0.00, 0,0,0])

  while (i < N):
    movel(pick_above, a=a, v=v)
    movel(pick, a=a, v=v*0.5)
    sleep(dt)
    movel(pick_above, a=a, v=v*0.5)

    movel(pal1_above, a=a, v=v)
    movel(pal1, a=a, v=v*0.5)
    sleep(dt)
    movel(pal1_above, a=a, v=v*0.5)

    movel(pick_above, a=a, v=v)
    movel(pick, a=a, v=v*0.5)
    sleep(dt)
    movel(pick_above, a=a, v=v*0.5)

    movel(pal2_above, a=a, v=v)
    movel(pal2, a=a, v=v*0.5)
    sleep(dt)
    movel(pal2_above, a=a, v=v*0.5)

    movel(pick_above, a=a, v=v)
    movel(pick, a=a, v=v*0.5)
    sleep(dt)
    movel(pick_above, a=a, v=v*0.5)

    movel(pal3_above, a=a, v=v)
    movel(pal3, a=a, v=v*0.5)
    sleep(dt)
    movel(pal3_above, a=a, v=v*0.5)

    movel(pick_above, a=a, v=v)
    movel(pick, a=a, v=v*0.5)
    sleep(dt)
    movel(pick_above, a=a, v=v*0.5)

    movel(pal4_above, a=a, v=v)
    movel(pal4, a=a, v=v*0.5)
    sleep(dt)
    movel(pal4_above, a=a, v=v*0.5)

    movel(home, a=a, v=v)

    i = i + 1
  end
end
task_t3()
"""

def save_fig(fig, name):
    fig.savefig(FIG_SUP / f"{name}.png")
    fig.savefig(FIG_SUP / f"{name}.pdf")

JOINT = ["Base (J0)","Shoulder (J1)","Elbow (J2)","Wrist 1 (J3)","Wrist 2 (J4)","Wrist 3 (J5)"]

# T3 baseline: 3kg gripper + 3kg pendant
# A2: extra mass ON TOP of 3kg baseline, pendant stays 3kg
# A3: DUCT TAPE only (electrical tape proven useless in NB5)
# A4: SKIPPED (failed on T1 and T2)
CONDITIONS = [
    # A2: Added mass — extra weight gripped alongside baseline 3kg
    {"anomaly": "A2", "severity": "3.5kg_gripper", "n_script": 40, "n_accept": 30,
     "payload_pendant_kg": 3.0, "payload_physical_kg": 3.5,
     "instruction": "A2 ADDED MASS — GRIPPER (+0.5 kg extra):\n"
                    "  - Grip baseline 3kg PLUS 0.5kg extra (total ~3.5 kg in gripper)\n"
                    "  - Pendant stays 3.0 kg (mismatch = 0.5 kg undeclared)"},

    {"anomaly": "A2", "severity": "4kg_gripper", "n_script": 40, "n_accept": 30,
     "payload_pendant_kg": 3.0, "payload_physical_kg": 4.0,
     "instruction": "A2 ADDED MASS — GRIPPER (+1 kg extra):\n"
                    "  - Swap: grip baseline 3kg PLUS 1kg extra (total ~4 kg)\n"
                    "  - Pendant stays 3.0 kg (mismatch = 1 kg undeclared)"},

    {"anomaly": "A2", "severity": "5kg_gripper", "n_script": 40, "n_accept": 30,
     "payload_pendant_kg": 3.0, "payload_physical_kg": 5.0,
     "instruction": "A2 ADDED MASS — GRIPPER (+2 kg extra):\n"
                    "  - Swap: grip baseline 3kg PLUS 2kg extra (total ~5 kg)\n"
                    "  - Pendant stays 3.0 kg (mismatch = 2 kg undeclared)\n"
                    "  - NOTE: UR5 max payload is 5kg — this is the limit!"},

    # A3: Friction — DUCT TAPE on J2 (2 levels)
    # NB4 duct tape: 10 wraps=ok, 17=ok, 29=protective stop
    # T3 has 3kg payload, large workspace — be conservative
    {"anomaly": "A3", "severity": "light_duct", "n_script": 40, "n_accept": 30,
     "payload_pendant_kg": 3.0, "payload_physical_kg": 3.0,
     "instruction": "A3 FRICTION (light DUCT tape):\n"
                    "  - Remove extra weights, back to 3 kg only in gripper\n"
                    "  - Wrap ~7 rounds of DUCT TAPE around J2 (elbow)\n"
                    "  - MUST be DUCT tape, NOT electrical tape!\n"
                    "  - NB4 reference: 10 wraps duct tape was fine on T1\n"
                    "  - NB5 lesson: electrical tape had ZERO effect"},

    {"anomaly": "A3", "severity": "medium_duct", "n_script": 40, "n_accept": 30,
     "payload_pendant_kg": 3.0, "payload_physical_kg": 3.0,
     "instruction": "A3 FRICTION (medium DUCT tape):\n"
                    "  - Add more duct tape: total ~14 rounds on J2\n"
                    "  - T3 has 3kg payload + large workspace — DO NOT go higher\n"
                    "  - If robot struggles or sounds strained, remove some tape"},

    # A5: TCP offset — grip extension piece alongside 3kg weight
    {"anomaly": "A5", "severity": "20mm", "n_script": 40, "n_accept": 30,
     "payload_pendant_kg": 3.0, "payload_physical_kg": 3.0,
     "instruction": "A5 TCP OFFSET (20 mm):\n"
                    "  - Remove all tape from J2\n"
                    "  - Grip short object (~20mm sticking out below fingertips)\n"
                    "  - Keep 3 kg weight in gripper too\n"
                    "  - Do NOT update TCP in pendant"},

    {"anomaly": "A5", "severity": "50mm", "n_script": 40, "n_accept": 30,
     "payload_pendant_kg": 3.0, "payload_physical_kg": 3.0,
     "instruction": "A5 TCP OFFSET (50 mm):\n"
                    "  - Swap to ~50mm object in gripper\n"
                    "  - Do NOT update TCP in pendant"},

    {"anomaly": "A5", "severity": "100mm", "n_script": 40, "n_accept": 30,
     "payload_pendant_kg": 3.0, "payload_physical_kg": 3.0,
     "instruction": "A5 TCP OFFSET (100 mm):\n"
                    "  - Swap to ~100mm object in gripper\n"
                    "  - Do NOT update TCP in pendant\n"
                    "  - T3 dips 8cm — extra 10cm offset may contact surface\n"
                    "  - Hand on E-STOP first few cycles"},
]

print(f"NB6: {TASK} anomaly collection — {len(CONDITIONS)} conditions, "
      f"{sum(c['n_script'] for c in CONDITIONS)} total cycles")
print(f"Baseline: 3kg gripper + 3kg pendant (same as NB3 healthy)")
print(f"A4 skipped (failed on T1 and T2)")
print(f"A3 uses DUCT TAPE (electrical tape failed in NB5)")

results = []

for idx, cond in enumerate(CONDITIONS):
    anom     = cond["anomaly"]
    sev      = cond["severity"]
    n_script = cond["n_script"]
    n_accept = cond["n_accept"]
    tag      = f"{anom}_{sev}"

    out_dir = ROOT / "Lab_Data" / "T3_Palletize" / anom
    out_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"  [{idx+1}/{len(CONDITIONS)}] {TASK} — {anom} ({sev})")
    print(f"  Cycles: {n_script} (accept >= {n_accept})")
    print(f"{'='*60}")
    print(f"\n{cond['instruction']}\n")

    input(f"Setup done? Press Enter to collect {tag} (E-STOP ready!)...")

    dt = 1.0 / HZ
    max_sec = n_script * 50 + 120
    nmax = int(max_sec * HZ)

    t_arr    = np.zeros(nmax)
    q_arr    = np.zeros((nmax, 6))
    qd_arr   = np.zeros((nmax, 6))
    cur_arr  = np.zeros((nmax, 6))
    tcp_arr  = np.zeros((nmax, 6))
    qdd_arr  = np.zeros((nmax, 6))
    mom_arr  = np.zeros((nmax, 6))
    tcpf_arr = np.zeros((nmax, 6))

    print("Logging started; sending URScript in 1.5s...")
    t0 = time.perf_counter()
    time.sleep(1.5)
    send_urscript(build_t3(n_script))
    print("URScript sent. Robot moving.\n")

    idle_ctr = 0
    idle_need = int(HZ * 4)
    min_run = 45.0
    last_print = t0

    i = 0
    while i < nmax:
        now = time.perf_counter()
        t_arr[i]   = now - t0
        q_arr[i]   = rtde_r.getActualQ()
        qd_arr[i]  = rtde_r.getActualQd()
        cur_arr[i] = rtde_r.getActualCurrent()
        tcp_arr[i] = rtde_r.getActualTCPPose()
        try:    qdd_arr[i]  = rtde_r.getTargetQdd()
        except: pass
        try:    mom_arr[i]  = rtde_r.getTargetMoment()
        except: pass
        try:    tcpf_arr[i] = rtde_r.getActualTCPForce()
        except: pass

        if now - last_print > 15.0:
            elapsed = now - t0
            v_now = np.max(np.abs(qd_arr[max(0, i-HZ):i+1]))
            i_now = np.max(np.abs(cur_arr[max(0, i-HZ):i+1]))
            print(f"  {elapsed:6.1f}s | samples: {i+1:>7,} | v_max: {v_now:.4f} | I_max: {i_now:.2f}A")
            last_print = now

        if t_arr[i] > min_run:
            if np.max(np.abs(qd_arr[i])) < 0.001:
                idle_ctr += 1
            else:
                idle_ctr = 0
            if idle_ctr >= idle_need:
                i += 1
                break

        target = (i + 1) * dt
        while (time.perf_counter() - t0) < target:
            pass
        i += 1

    time.sleep(1)
    send_urscript("stopj(2.0)\n")

    t_arr, q_arr, qd_arr = t_arr[:i], q_arr[:i], qd_arr[:i]
    cur_arr, tcp_arr = cur_arr[:i], tcp_arr[:i]
    qdd_arr, mom_arr, tcpf_arr = qdd_arr[:i], mom_arr[:i], tcpf_arr[:i]

    print(f"Collected {i:,} samples in {t_arr[-1]:.1f}s ({i/t_arr[-1]:.0f} Hz)")

    # Cycle detection with min_cycle_sec filter (T3 returns to HOME 7× per logical cycle)
    home = tcp_arr[0, :3]
    dist = np.sqrt(np.sum((tcp_arr[:, :3] - home)**2, axis=1)) * 1000

    n_base = min(int(2.0 * HZ), len(dist) // 10)
    noise = np.median(np.abs(np.diff(dist[:n_base]))) if n_base > 10 else 0.5
    home_r = max(10.0, 3.0 * noise)
    far_r  = 30.0

    cycle_num = np.zeros(len(tcp_arr), dtype=np.int32)
    cyc = 0
    was_far = False
    last_cycle_time = 0.0
    for j in range(len(tcp_arr)):
        if dist[j] > far_r:
            was_far = True
        if dist[j] < home_r and was_far:
            if (t_arr[j] - last_cycle_time) >= MIN_CYCLE_SEC:
                cyc += 1
                last_cycle_time = t_arr[j]
            was_far = False
        cycle_num[j] = cyc

    print(f"Detected {cyc} cycles (accept >= {n_accept}, min_cycle_sec={MIN_CYCLE_SEC})")

    has_qdd  = bool(np.any(qdd_arr != 0))
    has_mom  = bool(np.any(mom_arr != 0))
    has_tcpf = bool(np.any(tcpf_arr != 0))

    # Save HDF5
    ts_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    fname = f"UR5_T3_{anom}_{sev}_{cyc}cyc_{HZ}hz_{ts_str}.h5"
    fpath = out_dir / fname

    with h5py.File(str(fpath), "w") as f:
        for name, arr in [("timestamp", t_arr), ("actual_q", q_arr), ("actual_qd", qd_arr),
                          ("actual_current", cur_arr), ("actual_TCP_pose", tcp_arr),
                          ("target_qdd", qdd_arr), ("target_moment", mom_arr),
                          ("actual_TCP_force", tcpf_arr),
                          ("cycle_number", cycle_num)]:
            f.create_dataset(name, data=arr, compression="gzip")
        f.attrs.update({
            "task": TASK, "condition": "anomaly",
            "anomaly_type": anom, "severity": sev,
            "n_script": n_script, "n_detected": cyc,
            "hz": HZ, "duration_sec": float(t_arr[-1]),
            "payload_pendant_kg": cond["payload_pendant_kg"],
            "payload_physical_kg": cond["payload_physical_kg"],
            "v": V, "a": A, "notebook": NOTEBOOK,
            "has_target_qdd": has_qdd,
            "has_target_moment": has_mom,
            "has_actual_TCP_force": has_tcpf,
            "home_radius_mm": float(home_r),
            "min_cycle_sec": MIN_CYCLE_SEC,
        })

    print(f"Saved: {fpath} ({fpath.stat().st_size/1024/1024:.1f} MB)")

    # Session log
    log = {
        "notebook": NOTEBOOK, "timestamp": datetime.now().isoformat(),
        "task": TASK, "condition": "anomaly",
        "anomaly_type": anom, "severity": sev,
        "n_script": n_script, "n_detected": cyc, "hz": HZ,
        "duration_sec": round(float(t_arr[-1]), 1),
        "payload_pendant_kg": cond["payload_pendant_kg"],
        "payload_physical_kg": cond["payload_physical_kg"],
        "tcp_home": [round(x, 4) for x in tcp_arr[0].tolist()],
        "robot_mode": int(rtde_r.getRobotMode()),
        "safety_mode": int(rtde_r.getSafetyMode()),
        "has_target_qdd": has_qdd, "has_target_moment": has_mom,
        "has_actual_TCP_force": has_tcpf,
        "file": fname,
    }
    log_path = LOGS_DIR / f"{NOTEBOOK}_{anom}_{sev}_{ts_str}.json"
    with open(log_path, "w") as fl:
        json.dump(log, fl, indent=2)

    # Dataset CSV
    csv_exists = DATASET_LOG.exists()
    with open(DATASET_LOG, "a", newline="") as fl:
        w = csv.writer(fl)
        if not csv_exists:
            w.writerow(["filename","task","condition","anomaly","n_script","n_detected",
                         "duration_s","hz","payload_pendant_kg","payload_physical_kg",
                         "date","notebook","pass_fail"])
        w.writerow([fname, TASK, "anomaly", f"{anom}_{sev}", n_script, cyc,
                    round(t_arr[-1],1), HZ, cond["payload_pendant_kg"],
                    cond["payload_physical_kg"],
                    datetime.now().strftime("%Y-%m-%d %H:%M"), NOTEBOOK,
                    "PASS" if cyc >= n_accept else "FAIL"])

    # QC
    has_nan = any(np.any(np.isnan(arr)) for arr in [q_arr, qd_arr, cur_arr])
    active = sum(1 for j in range(6) if np.degrees(np.max(q_arr[:,j])-np.min(q_arr[:,j])) > 5)
    passed = cyc >= n_accept and not has_nan and active >= 3

    print(f"\nJoint currents:")
    for j in range(6):
        cc = cur_arr[:, j]
        print(f"  {JOINT[j]:16s}: mean={np.mean(cc):+.3f}A  "
              f"std={np.std(cc):.3f}A  [{np.min(cc):+.2f}, {np.max(cc):+.2f}]A")

    if passed:
        print(f"\nPASS: {cyc} cycles, {active} active joints")
    else:
        issues = []
        if cyc < n_accept: issues.append(f"Cycles: {cyc} < {n_accept}")
        if has_nan: issues.append("NaN")
        if active < 3: issues.append(f"Only {active} active joints")
        print(f"\nFAIL: {'; '.join(issues)} -> consider rerunning this condition")

    results.append({"tag": tag, "cycles": cyc, "pass": passed,
                    "I_max": float(np.max(np.abs(cur_arr))),
                    "J1_mean": float(np.mean(cur_arr[:,1])),
                    "J2_mean": float(np.mean(cur_arr[:,2]))})

    # Cycle overlay figure
    if cyc >= 3:
        fig, axes = plt.subplots(3, 2, figsize=FIG_TALL)
        fig.suptitle(f"T3 {anom} ({sev}): cycle overlay ({min(30,cyc)} cycles)",
                     fontweight="bold")
        for j in range(6):
            ax = axes[j//2, j%2]
            for ci in range(1, min(31, cyc+1)):
                cmask = cycle_num == ci
                if np.sum(cmask) < 10: continue
                ax.plot(cur_arr[cmask, j], alpha=0.15, lw=0.3)
            ax.set_title(JOINT[j])
            ax.set_ylabel("Current (A)")
            ax.set_xlabel("Sample index")
            ax.grid(True, alpha=0.2, lw=0.3)
            ax.text(-0.12, 1.0, chr(97+j), transform=ax.transAxes,
                    fontsize=8, fontweight="bold", va="top")
        plt.tight_layout(h_pad=0.5, w_pad=0.5)
        save_fig(fig, f"FigS_T3_{anom}_{sev}_cycle_overlay")
        plt.close(fig)

print(f"\n{'='*60}")
print(f"  NB6 COMPLETE — {TASK} anomaly collection")
print(f"{'='*60}")

print(f"\n{'Tag':<25s} {'Cycles':>6s} {'J1_mean':>8s} {'J2_mean':>8s} {'I_max':>6s} {'Pass':>5s}")
print("-" * 60)
for r in results:
    print(f"{r['tag']:<25s} {r['cycles']:>6d} {r['J1_mean']:>+8.3f} {r['J2_mean']:>+8.3f} "
          f"{r['I_max']:>6.2f} {'PASS' if r['pass'] else 'FAIL':>5s}")

n_pass = sum(1 for r in results if r["pass"])
print(f"\n{n_pass}/{len(results)} conditions passed")

print(f"\nNB3 healthy baseline for comparison:")
print(f"  J1=+2.612  J2=+3.347  I_max=6.53")

if n_pass == len(results):
    print("\nAll conditions passed! Data collection COMPLETE.")
    print("Next: NB7+ analysis notebooks (cycle segmentation, feature extraction, model training)")
else:
    failed = [r["tag"] for r in results if not r["pass"]]
    print(f"\nFailed: {', '.join(failed)} -> rerun those conditions manually")

In [ ]:
# NB5b6b — Combined remaining runs: NB6 reruns (T3) + NB5b (T2 A3 duct)
# NB6 FAILED because A2_5kg protective stop left robot in wrong position.
# All subsequent conditions got bad HOME → chain of failures.
# Fix: freedrive robot to correct HOME BEFORE starting.
# Strategy: HARDEST FIRST within each group.
# If A5_100mm passes → 50mm and 20mm will pass.
# If A3_medium_duct passes → light will pass.
# T3 block (pendant=3kg, gripper=3kg): 6 conditions
# T2 block (pendant=1kg, gripper=1kg): 2 conditions
# Total: 8 conditions ≈ 5 hrs (T3: ~35 min × 6 + T2: ~24 min × 2)

from rtde_receive import RTDEReceiveInterface

ROBOT_IP = "192.168.35.82"
rtde_r = RTDEReceiveInterface(ROBOT_IP)
print("Connected:", rtde_r.isConnected())

tcp0 = rtde_r.getActualTCPPose()
print(f"TCP: x={tcp0[0]:.3f}  y={tcp0[1]:.3f}  z={tcp0[2]:.3f}")
print()
print("CRITICAL: Is robot at correct HOME position?")
print("  NB6 good HOME was: x=-0.329  y=0.027  z=0.294")
print("  If not → freedrive to approximately this pose BEFORE continuing")
print()
print("Run order: HARDEST FIRST, tape grouped at end")
print("  NO-TAPE: T3 A5_100mm → A5_50mm → A5_20mm → A2_4.5kg")
print("  TAPE ON: T3 A3_medium → A3_light → T2 A3_medium → T2 A3_light")
print("           (tape goes on once, stays till last condition)")
print()
print("REMINDERS:")
print("  - DUCT TAPE for A3 (not electrical tape!)")
print("  - A2_5kg caused protective stop last time → trying 4.5kg instead")

import socket, time, json, csv
import numpy as np
import h5py
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime

matplotlib.rcParams.update({
    "font.family":       "Arial",
    "font.size":         6,
    "axes.titlesize":    7,
    "axes.labelsize":    6,
    "xtick.labelsize":   5,
    "ytick.labelsize":   5,
    "legend.fontsize":   5,
    "figure.titlesize":  7,
    "axes.linewidth":    0.5,
    "xtick.major.width": 0.4,
    "ytick.major.width": 0.4,
    "xtick.major.size":  2,
    "ytick.major.size":  2,
    "lines.linewidth":   0.4,
    "savefig.dpi":       1200,
    "savefig.bbox":      "tight",
    "savefig.pad_inches": 0.02,
    "pdf.fonttype":      42,
    "ps.fonttype":       42,
})

FIG_TALL = (180/25.4, 140/25.4)

ROOT        = Path(r"D:\Research\RCM")
LOGS_DIR    = ROOT / "Logs"
DATASET_LOG = ROOT / "Lab_Data" / "dataset_log.csv"
FIG_SUP     = ROOT / "Manuscript_Data" / "Figures" / "Supplementary"
for d in [LOGS_DIR, FIG_SUP]:
    d.mkdir(parents=True, exist_ok=True)

NOTEBOOK = "NB5b6b_remaining"
HZ       = 125
SCRIPT_PORT = 30002

def send_urscript(script: str):
    if not script.endswith("\n"):
        script += "\n"
    with socket.create_connection((ROBOT_IP, SCRIPT_PORT), timeout=5) as s:
        s.sendall(script.encode("utf-8"))

# ──────────────────────────────────────────────
# T3 PALLETIZING script
# ──────────────────────────────────────────────
V3 = 0.06;  A3_ACC = 0.25;  SLEEP3 = 0.3

def build_t3(N):
    return f"""
def task_t3():
  local v  = {V3}
  local a  = {A3_ACC}
  local dt = {SLEEP3}
  local N  = {N}
  local i  = 0

  local home = get_actual_tcp_pose()

  local pick_above = pose_trans(home, p[ 0.00,  0.00, -0.03, 0,0,0])
  local pick       = pose_trans(home, p[ 0.00,  0.00, -0.08, 0,0,0])

  local pal1 = pose_trans(home, p[ 0.18,  0.15, -0.05, 0,0,0])
  local pal2 = pose_trans(home, p[ 0.18, -0.15, -0.05, 0,0,0])
  local pal3 = pose_trans(home, p[-0.15,  0.15, -0.05, 0,0,0])
  local pal4 = pose_trans(home, p[-0.15, -0.15, -0.05, 0,0,0])

  local pal1_above = pose_trans(home, p[ 0.18,  0.15,  0.00, 0,0,0])
  local pal2_above = pose_trans(home, p[ 0.18, -0.15,  0.00, 0,0,0])
  local pal3_above = pose_trans(home, p[-0.15,  0.15,  0.00, 0,0,0])
  local pal4_above = pose_trans(home, p[-0.15, -0.15,  0.00, 0,0,0])

  while (i < N):
    movel(pick_above, a=a, v=v)
    movel(pick, a=a, v=v*0.5)
    sleep(dt)
    movel(pick_above, a=a, v=v*0.5)

    movel(pal1_above, a=a, v=v)
    movel(pal1, a=a, v=v*0.5)
    sleep(dt)
    movel(pal1_above, a=a, v=v*0.5)

    movel(pick_above, a=a, v=v)
    movel(pick, a=a, v=v*0.5)
    sleep(dt)
    movel(pick_above, a=a, v=v*0.5)

    movel(pal2_above, a=a, v=v)
    movel(pal2, a=a, v=v*0.5)
    sleep(dt)
    movel(pal2_above, a=a, v=v*0.5)

    movel(pick_above, a=a, v=v)
    movel(pick, a=a, v=v*0.5)
    sleep(dt)
    movel(pick_above, a=a, v=v*0.5)

    movel(pal3_above, a=a, v=v)
    movel(pal3, a=a, v=v*0.5)
    sleep(dt)
    movel(pal3_above, a=a, v=v*0.5)

    movel(pick_above, a=a, v=v)
    movel(pick, a=a, v=v*0.5)
    sleep(dt)
    movel(pick_above, a=a, v=v*0.5)

    movel(pal4_above, a=a, v=v)
    movel(pal4, a=a, v=v*0.5)
    sleep(dt)
    movel(pal4_above, a=a, v=v*0.5)

    movel(home, a=a, v=v)

    i = i + 1
  end
end
task_t3()
"""

# ──────────────────────────────────────────────
# T2 ASSEMBLY script
# ──────────────────────────────────────────────
V_FAST = 0.05;  V_SLOW = 0.02;  V_PRESS = 0.008;  A2_ACC = 0.15;  SLEEP2 = 0.4

def build_t2(N):
    return f"""
def task_t2():
  local vf = {V_FAST}
  local vs = {V_SLOW}
  local vp = {V_PRESS}
  local a  = {A2_ACC}
  local dt = {SLEEP2}
  local N  = {N}
  local i  = 0

  local home = get_actual_tcp_pose()

  local pick_above    = pose_trans(home, p[0.08, 0.0, 0.0, 0,0,0])
  local pick          = pose_trans(home, p[0.08, 0.0, -0.03, 0,0,0])

  local press1_above   = pose_trans(home, p[0.05, -0.06, 0.0, 0,0,0])
  local press1_contact = pose_trans(home, p[0.05, -0.06, -0.04, 0,0,0])
  local press1_deep    = pose_trans(home, p[0.05, -0.06, -0.07, 0,0,0])

  local press2_above   = pose_trans(home, p[0.05, 0.06, 0.0, 0,0,0])
  local press2_contact = pose_trans(home, p[0.05, 0.06, -0.04, 0,0,0])
  local press2_deep    = pose_trans(home, p[0.05, 0.06, -0.07, 0,0,0])

  while (i < N):
    movel(pick_above, a=a, v=vf)
    movel(pick, a=a, v=vs)
    sleep(0.3)
    movel(pick_above, a=a, v=vs)

    movel(press1_above, a=a, v=vf)
    movel(press1_contact, a=a, v=vs)
    movel(press1_deep, a=a, v=vp)
    sleep(dt)
    movel(press1_contact, a=a, v=vs)
    movel(press1_above, a=a, v=vf)

    movel(press2_above, a=a, v=vf)
    movel(press2_contact, a=a, v=vs)
    movel(press2_deep, a=a, v=vp)
    sleep(dt)
    movel(press2_contact, a=a, v=vs)
    movel(press2_above, a=a, v=vf)

    movel(pick_above, a=a, v=vf)
    movel(pick, a=a, v=vs)
    sleep(0.3)
    movel(pick_above, a=a, v=vs)
    movel(home, a=a, v=vf)

    i = i + 1
  end
end
task_t2()
"""

def save_fig(fig, name):
    fig.savefig(FIG_SUP / f"{name}.png")
    fig.savefig(FIG_SUP / f"{name}.pdf")

JOINT = ["Base (J0)","Shoulder (J1)","Elbow (J2)","Wrist 1 (J3)","Wrist 2 (J4)","Wrist 3 (J5)"]

# ──────────────────────────────────────────────
# CONDITIONS — hardest first within each task
# ──────────────────────────────────────────────

CONDITIONS = [
    # ══════ T3 NO-TAPE BLOCK — pendant 3kg, gripper 3kg ══════
    # A5: longest first (if 100mm passes, shorter ones will too)
    {"task": "T3", "anomaly": "A5", "severity": "100mm", "n_script": 40, "n_accept": 30,
     "payload_pendant_kg": 3.0, "payload_physical_kg": 3.0,
     "min_cycle_sec": 45, "build_fn": "t3",
     "out_subdir": "T3_Palletize/A5",
     "instruction": "═══ T3 NO-TAPE BLOCK — Set pendant to 3.0 kg ═══\n\n"
                    "A5 TCP OFFSET (100 mm) — HARDEST FIRST:\n"
                    "  - Grip ~100mm object + 3kg weight in gripper\n"
                    "  - Do NOT update TCP in pendant\n"
                    "  - T3 dips 8cm — extra 10cm = may contact surface\n"
                    "  - Hand on E-STOP for first few cycles\n"
                    "  - If this passes, 50mm and 20mm will be fine"},

    {"task": "T3", "anomaly": "A5", "severity": "50mm", "n_script": 40, "n_accept": 30,
     "payload_pendant_kg": 3.0, "payload_physical_kg": 3.0,
     "min_cycle_sec": 45, "build_fn": "t3",
     "out_subdir": "T3_Palletize/A5",
     "instruction": "A5 TCP OFFSET (50 mm):\n"
                    "  - Swap to ~50mm object in gripper\n"
                    "  - Do NOT update TCP in pendant"},

    {"task": "T3", "anomaly": "A5", "severity": "20mm", "n_script": 40, "n_accept": 30,
     "payload_pendant_kg": 3.0, "payload_physical_kg": 3.0,
     "min_cycle_sec": 45, "build_fn": "t3",
     "out_subdir": "T3_Palletize/A5",
     "instruction": "A5 TCP OFFSET (20 mm):\n"
                    "  - Swap to ~20mm object in gripper\n"
                    "  - Do NOT update TCP in pendant"},

    # A2: heaviest (4.5kg instead of 5kg — 5kg caused protective stop)
    {"task": "T3", "anomaly": "A2", "severity": "4.5kg_gripper", "n_script": 40, "n_accept": 30,
     "payload_pendant_kg": 3.0, "payload_physical_kg": 4.5,
     "min_cycle_sec": 45, "build_fn": "t3",
     "out_subdir": "T3_Palletize/A2",
     "instruction": "A2 ADDED MASS (4.5 kg total in gripper):\n"
                    "  - Remove A5 extension\n"
                    "  - Grip baseline 3kg PLUS 1.5kg extra (total ~4.5 kg)\n"
                    "  - Pendant stays 3.0 kg\n"
                    "  - 5kg caused protective stop last time — trying 4.5kg\n"
                    "  - Hand on E-STOP first few cycles\n"
                    "  - If this fails too, we keep only 2 A2 severity levels for T3"},

    # ══════ TAPE BLOCK — tape goes ON here, stays on till the end ══════
    # T3 A3: most tape first (if 14 wraps passes, 7 will too)
    {"task": "T3", "anomaly": "A3", "severity": "medium_duct", "n_script": 40, "n_accept": 30,
     "payload_pendant_kg": 3.0, "payload_physical_kg": 3.0,
     "min_cycle_sec": 45, "build_fn": "t3",
     "out_subdir": "T3_Palletize/A3",
     "instruction": "═══ TAPE BLOCK — Tape goes ON now, stays till end ═══\n\n"
                    "A3 FRICTION (medium DUCT tape) — HARDER FIRST:\n"
                    "  - Remove extra weight, back to 3kg only in gripper\n"
                    "  - Wrap ~14 rounds of DUCT TAPE on J2\n"
                    "  - MUST be DUCT tape, NOT electrical!\n"
                    "  - If robot struggles, remove some tape immediately\n"
                    "  - If this passes, 7 wraps will be fine"},

    {"task": "T3", "anomaly": "A3", "severity": "light_duct", "n_script": 40, "n_accept": 30,
     "payload_pendant_kg": 3.0, "payload_physical_kg": 3.0,
     "min_cycle_sec": 45, "build_fn": "t3",
     "out_subdir": "T3_Palletize/A3",
     "instruction": "A3 FRICTION (light DUCT tape):\n"
                    "  - Remove tape down to ~7 rounds on J2\n"
                    "  - Keep 3kg in gripper, pendant stays 3kg"},

    # T2 A3: keep tape on, just change pendant + gripper weight
    {"task": "T2", "anomaly": "A3", "severity": "medium_duct", "n_script": 40, "n_accept": 30,
     "payload_pendant_kg": 1.0, "payload_physical_kg": 1.0,
     "min_cycle_sec": 0, "build_fn": "t2",
     "out_subdir": "T2_Assembly/A3",
     "instruction": "═══ STILL IN TAPE BLOCK — keep tape, change weight ═══\n\n"
                    "A3 FRICTION (medium DUCT tape) for T2:\n"
                    "  - CHANGE pendant → 1.0 kg\n"
                    "  - CHANGE gripper → 1 kg only\n"
                    "  - Add tape back to ~14 rounds on J2 (if you removed for light)\n"
                    "  - Tape stays on from T3 — just adjust wraps if needed"},

    {"task": "T2", "anomaly": "A3", "severity": "light_duct", "n_script": 40, "n_accept": 30,
     "payload_pendant_kg": 1.0, "payload_physical_kg": 1.0,
     "min_cycle_sec": 0, "build_fn": "t2",
     "out_subdir": "T2_Assembly/A3",
     "instruction": "A3 FRICTION (light DUCT tape) for T2:\n"
                    "  - Remove tape down to ~7 rounds on J2\n"
                    "  - Keep 1kg in gripper, pendant 1kg\n"
                    "  - LAST CONDITION — remove all tape after this!"},
]

print(f"\nTotal: {len(CONDITIONS)} conditions")
print(f"  T3 block: {sum(1 for c in CONDITIONS if c['task']=='T3')} conditions (~3.5 hrs)")
print(f"  T2 block: {sum(1 for c in CONDITIONS if c['task']=='T2')} conditions (~50 min)")

results = []

for idx, cond in enumerate(CONDITIONS):
    task     = cond["task"]
    anom     = cond["anomaly"]
    sev      = cond["severity"]
    n_script = cond["n_script"]
    n_accept = cond["n_accept"]
    min_cyc  = cond["min_cycle_sec"]
    tag      = f"{task}_{anom}_{sev}"

    out_dir = ROOT / "Lab_Data" / cond["out_subdir"]
    out_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"  [{idx+1}/{len(CONDITIONS)}] {task} — {anom} ({sev})")
    print(f"  Cycles: {n_script} (accept >= {n_accept})")
    print(f"{'='*60}")
    print(f"\n{cond['instruction']}\n")

    # Check HOME before each condition
    tcp_now = rtde_r.getActualTCPPose()
    print(f"  Current TCP: x={tcp_now[0]:.3f}  y={tcp_now[1]:.3f}  z={tcp_now[2]:.3f}")

    input(f"Setup done? Press Enter to collect {tag} (E-STOP ready!)...")

    dt = 1.0 / HZ
    cycle_time_est = 51 if task == "T3" else 35
    max_sec = n_script * cycle_time_est + 120
    nmax = int(max_sec * HZ)

    t_arr    = np.zeros(nmax)
    q_arr    = np.zeros((nmax, 6))
    qd_arr   = np.zeros((nmax, 6))
    cur_arr  = np.zeros((nmax, 6))
    tcp_arr  = np.zeros((nmax, 6))
    qdd_arr  = np.zeros((nmax, 6))
    mom_arr  = np.zeros((nmax, 6))
    tcpf_arr = np.zeros((nmax, 6))

    build_fn = build_t3 if cond["build_fn"] == "t3" else build_t2

    print("Logging started; sending URScript in 1.5s...")
    t0 = time.perf_counter()
    time.sleep(1.5)
    send_urscript(build_fn(n_script))
    print("URScript sent. Robot moving.\n")

    idle_ctr = 0
    idle_need = int(HZ * 4)
    min_run = 45.0 if task == "T3" else 30.0
    last_print = t0

    i = 0
    while i < nmax:
        now = time.perf_counter()
        t_arr[i]   = now - t0
        q_arr[i]   = rtde_r.getActualQ()
        qd_arr[i]  = rtde_r.getActualQd()
        cur_arr[i] = rtde_r.getActualCurrent()
        tcp_arr[i] = rtde_r.getActualTCPPose()
        try:    qdd_arr[i]  = rtde_r.getTargetQdd()
        except: pass
        try:    mom_arr[i]  = rtde_r.getTargetMoment()
        except: pass
        try:    tcpf_arr[i] = rtde_r.getActualTCPForce()
        except: pass

        if now - last_print > 15.0:
            elapsed = now - t0
            v_now = np.max(np.abs(qd_arr[max(0, i-HZ):i+1]))
            i_now = np.max(np.abs(cur_arr[max(0, i-HZ):i+1]))
            print(f"  {elapsed:6.1f}s | samples: {i+1:>7,} | v_max: {v_now:.4f} | I_max: {i_now:.2f}A")
            last_print = now

        if t_arr[i] > min_run:
            if np.max(np.abs(qd_arr[i])) < 0.001:
                idle_ctr += 1
            else:
                idle_ctr = 0
            if idle_ctr >= idle_need:
                i += 1
                break

        target = (i + 1) * dt
        while (time.perf_counter() - t0) < target:
            pass
        i += 1

    time.sleep(1)
    send_urscript("stopj(2.0)\n")

    t_arr, q_arr, qd_arr = t_arr[:i], q_arr[:i], qd_arr[:i]
    cur_arr, tcp_arr = cur_arr[:i], tcp_arr[:i]
    qdd_arr, mom_arr, tcpf_arr = qdd_arr[:i], mom_arr[:i], tcpf_arr[:i]

    print(f"Collected {i:,} samples in {t_arr[-1]:.1f}s ({i/t_arr[-1]:.0f} Hz)")

    # Cycle detection
    home = tcp_arr[0, :3]
    dist = np.sqrt(np.sum((tcp_arr[:, :3] - home)**2, axis=1)) * 1000

    n_base = min(int(2.0 * HZ), len(dist) // 10)
    noise = np.median(np.abs(np.diff(dist[:n_base]))) if n_base > 10 else 0.5
    home_r = max(10.0, 3.0 * noise)
    far_r  = 30.0

    cycle_num = np.zeros(len(tcp_arr), dtype=np.int32)
    cyc = 0
    was_far = False
    last_cycle_time = 0.0
    for j in range(len(tcp_arr)):
        if dist[j] > far_r:
            was_far = True
        if dist[j] < home_r and was_far:
            if min_cyc == 0 or (t_arr[j] - last_cycle_time) >= min_cyc:
                cyc += 1
                last_cycle_time = t_arr[j]
            was_far = False
        cycle_num[j] = cyc

    if min_cyc > 0:
        print(f"Detected {cyc} cycles (accept >= {n_accept}, min_cycle_sec={min_cyc})")
    else:
        print(f"Detected {cyc} cycles (accept >= {n_accept})")

    has_qdd  = bool(np.any(qdd_arr != 0))
    has_mom  = bool(np.any(mom_arr != 0))
    has_tcpf = bool(np.any(tcpf_arr != 0))

    # Save HDF5
    ts_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    fname = f"UR5_{task}_{anom}_{sev}_{cyc}cyc_{HZ}hz_{ts_str}.h5"
    fpath = out_dir / fname

    with h5py.File(str(fpath), "w") as f:
        for name, arr in [("timestamp", t_arr), ("actual_q", q_arr), ("actual_qd", qd_arr),
                          ("actual_current", cur_arr), ("actual_TCP_pose", tcp_arr),
                          ("target_qdd", qdd_arr), ("target_moment", mom_arr),
                          ("actual_TCP_force", tcpf_arr),
                          ("cycle_number", cycle_num)]:
            f.create_dataset(name, data=arr, compression="gzip")
        f.attrs.update({
            "task": task, "condition": "anomaly",
            "anomaly_type": anom, "severity": sev,
            "n_script": n_script, "n_detected": cyc,
            "hz": HZ, "duration_sec": float(t_arr[-1]),
            "payload_pendant_kg": cond["payload_pendant_kg"],
            "payload_physical_kg": cond["payload_physical_kg"],
            "notebook": NOTEBOOK,
            "has_target_qdd": has_qdd,
            "has_target_moment": has_mom,
            "has_actual_TCP_force": has_tcpf,
            "home_radius_mm": float(home_r),
            "min_cycle_sec": min_cyc,
        })
        if anom == "A3":
            f.attrs["tape_type"] = "duct_tape"

    print(f"Saved: {fpath} ({fpath.stat().st_size/1024/1024:.1f} MB)")

    # Session log
    log = {
        "notebook": NOTEBOOK, "timestamp": datetime.now().isoformat(),
        "task": task, "condition": "anomaly",
        "anomaly_type": anom, "severity": sev,
        "n_script": n_script, "n_detected": cyc, "hz": HZ,
        "duration_sec": round(float(t_arr[-1]), 1),
        "payload_pendant_kg": cond["payload_pendant_kg"],
        "payload_physical_kg": cond["payload_physical_kg"],
        "tcp_home": [round(x, 4) for x in tcp_arr[0].tolist()],
        "robot_mode": int(rtde_r.getRobotMode()),
        "safety_mode": int(rtde_r.getSafetyMode()),
        "has_target_qdd": has_qdd, "has_target_moment": has_mom,
        "has_actual_TCP_force": has_tcpf,
        "file": fname,
    }
    if anom == "A3":
        log["tape_type"] = "duct_tape"
    log_path = LOGS_DIR / f"{NOTEBOOK}_{task}_{anom}_{sev}_{ts_str}.json"
    with open(log_path, "w") as fl:
        json.dump(log, fl, indent=2)

    # Dataset CSV
    csv_exists = DATASET_LOG.exists()
    with open(DATASET_LOG, "a", newline="") as fl:
        w = csv.writer(fl)
        if not csv_exists:
            w.writerow(["filename","task","condition","anomaly","n_script","n_detected",
                         "duration_s","hz","payload_pendant_kg","payload_physical_kg",
                         "date","notebook","pass_fail"])
        w.writerow([fname, task, "anomaly", f"{anom}_{sev}", n_script, cyc,
                    round(t_arr[-1],1), HZ, cond["payload_pendant_kg"],
                    cond["payload_physical_kg"],
                    datetime.now().strftime("%Y-%m-%d %H:%M"), NOTEBOOK,
                    "PASS" if cyc >= n_accept else "FAIL"])

    # QC
    has_nan = any(np.any(np.isnan(arr)) for arr in [q_arr, qd_arr, cur_arr])
    active = sum(1 for j in range(6) if np.degrees(np.max(q_arr[:,j])-np.min(q_arr[:,j])) > 5)
    passed = cyc >= n_accept and not has_nan and active >= 3

    print(f"\nJoint currents:")
    for j in range(6):
        cc = cur_arr[:, j]
        print(f"  {JOINT[j]:16s}: mean={np.mean(cc):+.3f}A  "
              f"std={np.std(cc):.3f}A  [{np.min(cc):+.2f}, {np.max(cc):+.2f}]A")

    if passed:
        print(f"\nPASS: {cyc} cycles, {active} active joints")
    else:
        issues = []
        if cyc < n_accept: issues.append(f"Cycles: {cyc} < {n_accept}")
        if has_nan: issues.append("NaN")
        if active < 3: issues.append(f"Only {active} active joints")
        print(f"\nFAIL: {'; '.join(issues)}")
        if anom == "A2" and "5" in sev:
            print("  → 5kg may be too heavy for T3 workspace. Consider keeping 2 severity levels only.")

    results.append({"tag": tag, "task": task, "cycles": cyc, "pass": passed,
                    "I_max": float(np.max(np.abs(cur_arr))),
                    "J1_mean": float(np.mean(cur_arr[:,1])),
                    "J2_mean": float(np.mean(cur_arr[:,2])),
                    "J2_std": float(np.std(cur_arr[:,2]))})

    # Cycle overlay figure
    if cyc >= 3:
        fig, axes = plt.subplots(3, 2, figsize=FIG_TALL)
        fig.suptitle(f"{task} {anom} ({sev}): cycle overlay ({min(30,cyc)} cycles)",
                     fontweight="bold")
        for j in range(6):
            ax = axes[j//2, j%2]
            for ci in range(1, min(31, cyc+1)):
                cmask = cycle_num == ci
                if np.sum(cmask) < 10: continue
                ax.plot(cur_arr[cmask, j], alpha=0.15, lw=0.3)
            ax.set_title(JOINT[j])
            ax.set_ylabel("Current (A)")
            ax.set_xlabel("Sample index")
            ax.grid(True, alpha=0.2, lw=0.3)
            ax.text(-0.12, 1.0, chr(97+j), transform=ax.transAxes,
                    fontsize=8, fontweight="bold", va="top")
        plt.tight_layout(h_pad=0.5, w_pad=0.5)
        save_fig(fig, f"FigS_{task}_{anom}_{sev}_cycle_overlay")
        plt.close(fig)

print(f"\n{'='*60}")
print(f"  ALL REMAINING RUNS COMPLETE")
print(f"{'='*60}")

print(f"\n{'Tag':<30s} {'Cycles':>6s} {'J1_mean':>8s} {'J2_mean':>8s} {'J2_std':>7s} {'I_max':>6s} {'Pass':>5s}")
print("-" * 75)
last_task = ""
for r in results:
    if r["task"] != last_task:
        if last_task: print()
        last_task = r["task"]
    print(f"{r['tag']:<30s} {r['cycles']:>6d} {r['J1_mean']:>+8.3f} {r['J2_mean']:>+8.3f} "
          f"{r['J2_std']:>7.3f} {r['I_max']:>6.2f} {'PASS' if r['pass'] else 'FAIL':>5s}")

n_pass = sum(1 for r in results if r["pass"])
print(f"\n{n_pass}/{len(results)} conditions passed")

print(f"\nBaseline references:")
print(f"  NB3 T3 healthy: J1=+2.612  J2=+3.347  J2_std=0.668  I_max=6.53")
print(f"  NB2 T2 healthy: J1=+3.537  J2=+2.655  J2_std=0.601  I_max=5.10")

t3_pass = sum(1 for r in results if r["task"]=="T3" and r["pass"])
t2_pass = sum(1 for r in results if r["task"]=="T2" and r["pass"])
t3_total = sum(1 for r in results if r["task"]=="T3")
t2_total = sum(1 for r in results if r["task"]=="T2")
print(f"\nT3: {t3_pass}/{t3_total} passed")
print(f"T2: {t2_pass}/{t2_total} passed")

if n_pass == len(results):
    print("\nAll remaining runs passed! Experiment 1 data collection COMPLETE.")
    print("Next steps:")
    print("  1. Update lab notebook with NB5b6b results")
    print("  2. Define T4-T6 trajectories for Experiment 2")
    print("  3. Begin model development (PI-SBD + baselines)")

In [ ]:
# NB5b6b_tape — Final remaining runs: A3 duct tape for T3 + T2
# 14 wraps: T3 (3kg) → T2 (1kg)   [tape stays, change pendant+gripper]
# 7 wraps: T2 (1kg) → T3 (3kg)   [peel tape, then change pendant+gripper]
# Completes ALL remaining NB5b (T2 A3 duct) and NB6 (T3 A3 duct)
# WARNING: 14 wraps on T3 (3kg, large workspace) may cause protective stop.
# Previous attempt likely crashed here. Hand on E-STOP!

from rtde_receive import RTDEReceiveInterface

ROBOT_IP = "192.168.35.82"
rtde_r = RTDEReceiveInterface(ROBOT_IP)
print("Connected:", rtde_r.isConnected())

tcp0 = rtde_r.getActualTCPPose()
print(f"TCP: x={tcp0[0]:.3f}  y={tcp0[1]:.3f}  z={tcp0[2]:.3f}")
print()
print("4 conditions: ~2.5 hrs total")
print("  [1] 14 wraps + 3kg (T3)  ← HARDEST, may protective stop")
print("  [2] 14 wraps + 1kg (T2)  ← change pendant+gripper only")
print("  [3]  7 wraps + 1kg (T2)  ← peel half tape")
print("  [4]  7 wraps + 3kg (T3)  ← change pendant+gripper only")
print()
print("DUCT TAPE ONLY. NOT electrical tape!")

import socket, time, json, csv
import numpy as np
import h5py
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime

matplotlib.rcParams.update({
    "font.family":       "Arial",
    "font.size":         6,
    "axes.titlesize":    7,
    "axes.labelsize":    6,
    "xtick.labelsize":   5,
    "ytick.labelsize":   5,
    "legend.fontsize":   5,
    "figure.titlesize":  7,
    "axes.linewidth":    0.5,
    "xtick.major.width": 0.4,
    "ytick.major.width": 0.4,
    "xtick.major.size":  2,
    "ytick.major.size":  2,
    "lines.linewidth":   0.4,
    "savefig.dpi":       1200,
    "savefig.bbox":      "tight",
    "savefig.pad_inches": 0.02,
    "pdf.fonttype":      42,
    "ps.fonttype":       42,
})

FIG_TALL = (180/25.4, 140/25.4)

ROOT        = Path(r"D:\Research\RCM")
LOGS_DIR    = ROOT / "Logs"
DATASET_LOG = ROOT / "Lab_Data" / "dataset_log.csv"
FIG_SUP     = ROOT / "Manuscript_Data" / "Figures" / "Supplementary"
for d in [LOGS_DIR, FIG_SUP]:
    d.mkdir(parents=True, exist_ok=True)

NOTEBOOK = "NB5b6b_tape"
HZ       = 125
SCRIPT_PORT = 30002

def send_urscript(script: str):
    if not script.endswith("\n"):
        script += "\n"
    with socket.create_connection((ROBOT_IP, SCRIPT_PORT), timeout=5) as s:
        s.sendall(script.encode("utf-8"))

# ── T3 PALLETIZING ──
V3 = 0.06;  A3_ACC = 0.25;  SLEEP3 = 0.3

def build_t3(N):
    return f"""
def task_t3():
  local v  = {V3}
  local a  = {A3_ACC}
  local dt = {SLEEP3}
  local N  = {N}
  local i  = 0

  local home = get_actual_tcp_pose()

  local pick_above = pose_trans(home, p[ 0.00,  0.00, -0.03, 0,0,0])
  local pick       = pose_trans(home, p[ 0.00,  0.00, -0.08, 0,0,0])

  local pal1 = pose_trans(home, p[ 0.18,  0.15, -0.05, 0,0,0])
  local pal2 = pose_trans(home, p[ 0.18, -0.15, -0.05, 0,0,0])
  local pal3 = pose_trans(home, p[-0.15,  0.15, -0.05, 0,0,0])
  local pal4 = pose_trans(home, p[-0.15, -0.15, -0.05, 0,0,0])

  local pal1_above = pose_trans(home, p[ 0.18,  0.15,  0.00, 0,0,0])
  local pal2_above = pose_trans(home, p[ 0.18, -0.15,  0.00, 0,0,0])
  local pal3_above = pose_trans(home, p[-0.15,  0.15,  0.00, 0,0,0])
  local pal4_above = pose_trans(home, p[-0.15, -0.15,  0.00, 0,0,0])

  while (i < N):
    movel(pick_above, a=a, v=v)
    movel(pick, a=a, v=v*0.5)
    sleep(dt)
    movel(pick_above, a=a, v=v*0.5)

    movel(pal1_above, a=a, v=v)
    movel(pal1, a=a, v=v*0.5)
    sleep(dt)
    movel(pal1_above, a=a, v=v*0.5)

    movel(pick_above, a=a, v=v)
    movel(pick, a=a, v=v*0.5)
    sleep(dt)
    movel(pick_above, a=a, v=v*0.5)

    movel(pal2_above, a=a, v=v)
    movel(pal2, a=a, v=v*0.5)
    sleep(dt)
    movel(pal2_above, a=a, v=v*0.5)

    movel(pick_above, a=a, v=v)
    movel(pick, a=a, v=v*0.5)
    sleep(dt)
    movel(pick_above, a=a, v=v*0.5)

    movel(pal3_above, a=a, v=v)
    movel(pal3, a=a, v=v*0.5)
    sleep(dt)
    movel(pal3_above, a=a, v=v*0.5)

    movel(pick_above, a=a, v=v)
    movel(pick, a=a, v=v*0.5)
    sleep(dt)
    movel(pick_above, a=a, v=v*0.5)

    movel(pal4_above, a=a, v=v)
    movel(pal4, a=a, v=v*0.5)
    sleep(dt)
    movel(pal4_above, a=a, v=v*0.5)

    movel(home, a=a, v=v)

    i = i + 1
  end
end
task_t3()
"""

# ── T2 ASSEMBLY ──
V_FAST = 0.05;  V_SLOW = 0.02;  V_PRESS = 0.008;  A2_ACC = 0.15;  SLEEP2 = 0.4

def build_t2(N):
    return f"""
def task_t2():
  local vf = {V_FAST}
  local vs = {V_SLOW}
  local vp = {V_PRESS}
  local a  = {A2_ACC}
  local dt = {SLEEP2}
  local N  = {N}
  local i  = 0

  local home = get_actual_tcp_pose()

  local pick_above    = pose_trans(home, p[0.08, 0.0, 0.0, 0,0,0])
  local pick          = pose_trans(home, p[0.08, 0.0, -0.03, 0,0,0])

  local press1_above   = pose_trans(home, p[0.05, -0.06, 0.0, 0,0,0])
  local press1_contact = pose_trans(home, p[0.05, -0.06, -0.04, 0,0,0])
  local press1_deep    = pose_trans(home, p[0.05, -0.06, -0.07, 0,0,0])

  local press2_above   = pose_trans(home, p[0.05, 0.06, 0.0, 0,0,0])
  local press2_contact = pose_trans(home, p[0.05, 0.06, -0.04, 0,0,0])
  local press2_deep    = pose_trans(home, p[0.05, 0.06, -0.07, 0,0,0])

  while (i < N):
    movel(pick_above, a=a, v=vf)
    movel(pick, a=a, v=vs)
    sleep(0.3)
    movel(pick_above, a=a, v=vs)

    movel(press1_above, a=a, v=vf)
    movel(press1_contact, a=a, v=vs)
    movel(press1_deep, a=a, v=vp)
    sleep(dt)
    movel(press1_contact, a=a, v=vs)
    movel(press1_above, a=a, v=vf)

    movel(press2_above, a=a, v=vf)
    movel(press2_contact, a=a, v=vs)
    movel(press2_deep, a=a, v=vp)
    sleep(dt)
    movel(press2_contact, a=a, v=vs)
    movel(press2_above, a=a, v=vf)

    movel(pick_above, a=a, v=vf)
    movel(pick, a=a, v=vs)
    sleep(0.3)
    movel(pick_above, a=a, v=vs)
    movel(home, a=a, v=vf)

    i = i + 1
  end
end
task_t2()
"""

def save_fig(fig, name):
    fig.savefig(FIG_SUP / f"{name}.png")
    fig.savefig(FIG_SUP / f"{name}.pdf")

JOINT = ["Base (J0)","Shoulder (J1)","Elbow (J2)","Wrist 1 (J3)","Wrist 2 (J4)","Wrist 3 (J5)"]

# ── CONDITIONS: minimize tape + pendant changes ──
CONDITIONS = [
    # 14 wraps block
    {"task": "T3", "anomaly": "A3", "severity": "medium_duct", "n_script": 40, "n_accept": 30,
     "payload_pendant_kg": 3.0, "payload_physical_kg": 3.0,
     "min_cycle_sec": 45, "build_fn": "t3",
     "out_subdir": "T3_Palletize/A3",
     "instruction": "═══ 14 WRAPS BLOCK ═══\n\n"
                    "[1/4] T3 A3 — 14 wraps DUCT TAPE + 3kg\n"
                    "  - Pendant: 3.0 kg\n"
                    "  - Gripper: 3.0 kg\n"
                    "  - Wrap ~14 rounds of DUCT TAPE on J2\n"
                    "  - ⚠️ HARDEST CONDITION — previous attempt likely crashed here\n"
                    "  - Hand on E-STOP! If protective stop → try 10 wraps instead\n"
                    "  - MUST be DUCT tape, NOT electrical!"},

    {"task": "T2", "anomaly": "A3", "severity": "medium_duct", "n_script": 40, "n_accept": 30,
     "payload_pendant_kg": 1.0, "payload_physical_kg": 1.0,
     "min_cycle_sec": 0, "build_fn": "t2",
     "out_subdir": "T2_Assembly/A3",
     "instruction": "[2/4] T2 A3 — 14 wraps DUCT TAPE + 1kg\n"
                    "  - ⚠️ CHANGE pendant → 1.0 kg\n"
                    "  - ⚠️ CHANGE gripper → 1.0 kg\n"
                    "  - Keep tape at 14 wraps — do NOT touch tape"},

    # 7 wraps block
    {"task": "T2", "anomaly": "A3", "severity": "light_duct", "n_script": 40, "n_accept": 30,
     "payload_pendant_kg": 1.0, "payload_physical_kg": 1.0,
     "min_cycle_sec": 0, "build_fn": "t2",
     "out_subdir": "T2_Assembly/A3",
     "instruction": "═══ 7 WRAPS BLOCK ═══\n\n"
                    "[3/4] T2 A3 — 7 wraps DUCT TAPE + 1kg\n"
                    "  - Peel tape down to ~7 wraps on J2\n"
                    "  - Pendant stays 1.0 kg\n"
                    "  - Gripper stays 1.0 kg"},

    {"task": "T3", "anomaly": "A3", "severity": "light_duct", "n_script": 40, "n_accept": 30,
     "payload_pendant_kg": 3.0, "payload_physical_kg": 3.0,
     "min_cycle_sec": 45, "build_fn": "t3",
     "out_subdir": "T3_Palletize/A3",
     "instruction": "[4/4] T3 A3 — 7 wraps DUCT TAPE + 3kg\n"
                    "  - ⚠️ CHANGE pendant → 3.0 kg\n"
                    "  - ⚠️ CHANGE gripper → 3.0 kg\n"
                    "  - Keep tape at 7 wraps — do NOT touch tape\n"
                    "  - LAST CONDITION — remove all tape after this!"},
]

print(f"\nTotal: {len(CONDITIONS)} conditions")
print(f"Tape changes: 1 (14→7 between condition 2 and 3)")
print(f"Pendant changes: 2 (3→1 between 1-2, 1→3 between 3-4)")

results = []

for idx, cond in enumerate(CONDITIONS):
    task     = cond["task"]
    anom     = cond["anomaly"]
    sev      = cond["severity"]
    n_script = cond["n_script"]
    n_accept = cond["n_accept"]
    min_cyc  = cond["min_cycle_sec"]
    tag      = f"{task}_{anom}_{sev}"

    out_dir = ROOT / "Lab_Data" / cond["out_subdir"]
    out_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"  [{idx+1}/{len(CONDITIONS)}] {task} — {anom} ({sev})")
    print(f"  Cycles: {n_script} (accept >= {n_accept})")
    print(f"{'='*60}")
    print(f"\n{cond['instruction']}\n")

    tcp_now = rtde_r.getActualTCPPose()
    print(f"  Current TCP: x={tcp_now[0]:.3f}  y={tcp_now[1]:.3f}  z={tcp_now[2]:.3f}")

    input(f"Setup done? Press Enter to collect {tag} (E-STOP ready!)...")

    # Reconnect RTDE in case previous protective stop killed connection
    try:
        rtde_r.getActualQ()
    except:
        print("  RTDE disconnected — reconnecting...")
        rtde_r = RTDEReceiveInterface(ROBOT_IP)
        print(f"  Reconnected: {rtde_r.isConnected()}")

    dt = 1.0 / HZ
    cycle_time_est = 51 if task == "T3" else 35
    max_sec = n_script * cycle_time_est + 120
    nmax = int(max_sec * HZ)

    t_arr    = np.zeros(nmax)
    q_arr    = np.zeros((nmax, 6))
    qd_arr   = np.zeros((nmax, 6))
    cur_arr  = np.zeros((nmax, 6))
    tcp_arr  = np.zeros((nmax, 6))
    qdd_arr  = np.zeros((nmax, 6))
    mom_arr  = np.zeros((nmax, 6))
    tcpf_arr = np.zeros((nmax, 6))

    build_fn = build_t3 if cond["build_fn"] == "t3" else build_t2

    print("Logging started; sending URScript in 1.5s...")
    t0 = time.perf_counter()
    time.sleep(1.5)
    send_urscript(build_fn(n_script))
    print("URScript sent. Robot moving.\n")

    idle_ctr = 0
    idle_need = int(HZ * 4)
    min_run = 45.0 if task == "T3" else 30.0
    last_print = t0

    i = 0
    try:
        while i < nmax:
            now = time.perf_counter()
            t_arr[i]   = now - t0
            q_arr[i]   = rtde_r.getActualQ()
            qd_arr[i]  = rtde_r.getActualQd()
            cur_arr[i] = rtde_r.getActualCurrent()
            tcp_arr[i] = rtde_r.getActualTCPPose()
            try:    qdd_arr[i]  = rtde_r.getTargetQdd()
            except: pass
            try:    mom_arr[i]  = rtde_r.getTargetMoment()
            except: pass
            try:    tcpf_arr[i] = rtde_r.getActualTCPForce()
            except: pass

            if now - last_print > 15.0:
                elapsed = now - t0
                v_now = np.max(np.abs(qd_arr[max(0, i-HZ):i+1]))
                i_now = np.max(np.abs(cur_arr[max(0, i-HZ):i+1]))

                # Check safety mode
                try:
                    smode = rtde_r.getSafetyMode()
                    smode_str = f" | safety={smode}" if smode != 1 else ""
                except:
                    smode_str = " | safety=?"

                print(f"  {elapsed:6.1f}s | samples: {i+1:>7,} | v_max: {v_now:.4f} | I_max: {i_now:.2f}A{smode_str}")
                last_print = now

            if t_arr[i] > min_run:
                if np.max(np.abs(qd_arr[i])) < 0.001:
                    idle_ctr += 1
                else:
                    idle_ctr = 0
                if idle_ctr >= idle_need:
                    i += 1
                    break

            target = (i + 1) * dt
            while (time.perf_counter() - t0) < target:
                pass
            i += 1

    except Exception as e:
        print(f"\n  ⚠️ COLLECTION INTERRUPTED: {e}")
        print(f"  Likely protective stop. Saving {i} samples collected so far.")

    time.sleep(1)
    try:
        send_urscript("stopj(2.0)\n")
    except:
        pass

    t_arr, q_arr, qd_arr = t_arr[:i], q_arr[:i], qd_arr[:i]
    cur_arr, tcp_arr = cur_arr[:i], tcp_arr[:i]
    qdd_arr, mom_arr, tcpf_arr = qdd_arr[:i], mom_arr[:i], tcpf_arr[:i]

    dur = t_arr[-1] if len(t_arr) > 0 else 0
    print(f"Collected {i:,} samples in {dur:.1f}s ({i/dur:.0f} Hz)" if dur > 0 else "No samples collected")

    # Cycle detection
    if i > 100:
        home = tcp_arr[0, :3]
        dist = np.sqrt(np.sum((tcp_arr[:, :3] - home)**2, axis=1)) * 1000

        n_base = min(int(2.0 * HZ), len(dist) // 10)
        noise = np.median(np.abs(np.diff(dist[:n_base]))) if n_base > 10 else 0.5
        home_r = max(10.0, 3.0 * noise)
        far_r  = 30.0

        cycle_num = np.zeros(len(tcp_arr), dtype=np.int32)
        cyc = 0
        was_far = False
        last_cycle_time = 0.0
        for j in range(len(tcp_arr)):
            if dist[j] > far_r:
                was_far = True
            if dist[j] < home_r and was_far:
                if min_cyc == 0 or (t_arr[j] - last_cycle_time) >= min_cyc:
                    cyc += 1
                    last_cycle_time = t_arr[j]
                was_far = False
            cycle_num[j] = cyc
    else:
        cyc = 0
        cycle_num = np.zeros(max(1, i), dtype=np.int32)
        home_r = 10.0

    print(f"Detected {cyc} cycles (accept >= {n_accept})")

    has_qdd  = bool(np.any(qdd_arr != 0)) if i > 0 else False
    has_mom  = bool(np.any(mom_arr != 0)) if i > 0 else False
    has_tcpf = bool(np.any(tcpf_arr != 0)) if i > 0 else False

    # Save HDF5
    ts_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    fname = f"UR5_{task}_{anom}_{sev}_{cyc}cyc_{HZ}hz_{ts_str}.h5"
    fpath = out_dir / fname

    with h5py.File(str(fpath), "w") as f:
        for name, arr in [("timestamp", t_arr), ("actual_q", q_arr), ("actual_qd", qd_arr),
                          ("actual_current", cur_arr), ("actual_TCP_pose", tcp_arr),
                          ("target_qdd", qdd_arr), ("target_moment", mom_arr),
                          ("actual_TCP_force", tcpf_arr),
                          ("cycle_number", cycle_num)]:
            f.create_dataset(name, data=arr, compression="gzip")
        f.attrs.update({
            "task": task, "condition": "anomaly",
            "anomaly_type": anom, "severity": sev,
            "tape_type": "duct_tape",
            "n_script": n_script, "n_detected": cyc,
            "hz": HZ, "duration_sec": float(dur),
            "payload_pendant_kg": cond["payload_pendant_kg"],
            "payload_physical_kg": cond["payload_physical_kg"],
            "notebook": NOTEBOOK,
            "has_target_qdd": has_qdd,
            "has_target_moment": has_mom,
            "has_actual_TCP_force": has_tcpf,
            "home_radius_mm": float(home_r),
            "min_cycle_sec": min_cyc,
        })

    print(f"Saved: {fpath} ({fpath.stat().st_size/1024/1024:.1f} MB)")

    # Session log
    log = {
        "notebook": NOTEBOOK, "timestamp": datetime.now().isoformat(),
        "task": task, "condition": "anomaly",
        "anomaly_type": anom, "severity": sev,
        "tape_type": "duct_tape",
        "n_script": n_script, "n_detected": cyc, "hz": HZ,
        "duration_sec": round(float(dur), 1),
        "payload_pendant_kg": cond["payload_pendant_kg"],
        "payload_physical_kg": cond["payload_physical_kg"],
        "tcp_home": [round(x, 4) for x in tcp_arr[0].tolist()] if i > 0 else [],
        "file": fname,
    }
    try:
        log["robot_mode"] = int(rtde_r.getRobotMode())
        log["safety_mode"] = int(rtde_r.getSafetyMode())
    except:
        log["robot_mode"] = -1
        log["safety_mode"] = -1

    log_path = LOGS_DIR / f"{NOTEBOOK}_{task}_{anom}_{sev}_{ts_str}.json"
    with open(log_path, "w") as fl:
        json.dump(log, fl, indent=2)

    # Dataset CSV
    csv_exists = DATASET_LOG.exists()
    with open(DATASET_LOG, "a", newline="") as fl:
        w = csv.writer(fl)
        if not csv_exists:
            w.writerow(["filename","task","condition","anomaly","n_script","n_detected",
                         "duration_s","hz","payload_pendant_kg","payload_physical_kg",
                         "date","notebook","pass_fail"])
        w.writerow([fname, task, "anomaly", f"{anom}_{sev}", n_script, cyc,
                    round(dur,1), HZ, cond["payload_pendant_kg"],
                    cond["payload_physical_kg"],
                    datetime.now().strftime("%Y-%m-%d %H:%M"), NOTEBOOK,
                    "PASS" if cyc >= n_accept else "FAIL"])

    # QC
    has_nan = any(np.any(np.isnan(arr)) for arr in [q_arr, qd_arr, cur_arr]) if i > 0 else True
    active = sum(1 for j in range(6) if np.degrees(np.max(q_arr[:,j])-np.min(q_arr[:,j])) > 5) if i > 100 else 0
    passed = cyc >= n_accept and not has_nan and active >= 3

    print(f"\nJoint currents:")
    for j in range(6):
        cc = cur_arr[:, j]
        if len(cc) > 0:
            print(f"  {JOINT[j]:16s}: mean={np.mean(cc):+.3f}A  "
                  f"std={np.std(cc):.3f}A  [{np.min(cc):+.2f}, {np.max(cc):+.2f}]A")

    if passed:
        print(f"\nPASS: {cyc} cycles, {active} active joints")
    else:
        issues = []
        if cyc < n_accept: issues.append(f"Cycles: {cyc} < {n_accept}")
        if has_nan: issues.append("NaN")
        if active < 3: issues.append(f"Only {active} active joints")
        print(f"\nFAIL: {'; '.join(issues)}")
        if task == "T3" and sev == "medium_duct":
            print("  → If protective stop: try 10 wraps instead of 14")
            print("  → Then continue with conditions 2-4")

    results.append({"tag": tag, "task": task, "cycles": cyc, "pass": passed,
                    "I_max": float(np.max(np.abs(cur_arr))) if i > 0 else 0,
                    "J1_mean": float(np.mean(cur_arr[:,1])) if i > 0 else 0,
                    "J2_mean": float(np.mean(cur_arr[:,2])) if i > 0 else 0,
                    "J2_std": float(np.std(cur_arr[:,2])) if i > 0 else 0})

    # Cycle overlay figure
    if cyc >= 3:
        fig, axes = plt.subplots(3, 2, figsize=FIG_TALL)
        fig.suptitle(f"{task} {anom} ({sev}): cycle overlay ({min(30,cyc)} cycles)",
                     fontweight="bold")
        for j in range(6):
            ax = axes[j//2, j%2]
            for ci in range(1, min(31, cyc+1)):
                cmask = cycle_num == ci
                if np.sum(cmask) < 10: continue
                ax.plot(cur_arr[cmask, j], alpha=0.15, lw=0.3)
            ax.set_title(JOINT[j])
            ax.set_ylabel("Current (A)")
            ax.set_xlabel("Sample index")
            ax.grid(True, alpha=0.2, lw=0.3)
            ax.text(-0.12, 1.0, chr(97+j), transform=ax.transAxes,
                    fontsize=8, fontweight="bold", va="top")
        plt.tight_layout(h_pad=0.5, w_pad=0.5)
        save_fig(fig, f"FigS_{task}_{anom}_{sev}_cycle_overlay")
        plt.close(fig)

print(f"\n{'='*60}")
print(f"  ALL TAPE CONDITIONS COMPLETE")
print(f"{'='*60}")

print(f"\n{'Tag':<30s} {'Cycles':>6s} {'J1_mean':>8s} {'J2_mean':>8s} {'J2_std':>7s} {'I_max':>6s} {'Pass':>5s}")
print("-" * 75)
for r in results:
    print(f"{r['tag']:<30s} {r['cycles']:>6d} {r['J1_mean']:>+8.3f} {r['J2_mean']:>+8.3f} "
          f"{r['J2_std']:>7.3f} {r['I_max']:>6.2f} {'PASS' if r['pass'] else 'FAIL':>5s}")

n_pass = sum(1 for r in results if r["pass"])
print(f"\n{n_pass}/{len(results)} conditions passed")

print(f"\nBaseline references:")
print(f"  NB3 T3 healthy: J1=+2.612  J2=+3.347  J2_std=0.668")
print(f"  NB2 T2 healthy: J1=+3.537  J2=+2.655  J2_std=0.601")
print(f"  NB4 T1 duct 10 wraps: J2_std=1.649  (3× baseline)")

if n_pass == len(results):
    print("\n✅ All tape conditions passed!")
    print("NB5b (T2 A3 duct) and NB6 (T3 A3 duct) COMPLETE.")
    print("Experiment 1 data collection is DONE.")